# E4 NF Geometry Debug

Fast, single-condition debug notebook for the E4 tiny-MLP weight-space experiment.

It does not run LR tuning, downstream optimization, or E0-E3. It only trains one RealNVP NF on the E4 geometry objective and checks whether the held-out pullback geometry improves.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display

from post_train_research.loss_landscape_analysis.flow_preconditioning.e4_debug import (
    E4DebugConfig,
    build_e4_debug_state,
    train_e4_debug_flow,
)

plt.rcParams['figure.dpi'] = 130


## Hardcoded Debug Config

Defaults are intentionally small so the notebook can be rerun while debugging. Increase `FLOW_STEPS`, `FLOW_RANDOM_SAMPLES`, or eval sample counts only after the fast run gives a clear signal.


In [ ]:
RUN_LABEL = 'e4_nf_geometry_debug_seed0_rho1e2_fast'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

DEBUG_CFG = E4DebugConfig(
    run_label=RUN_LABEL,
    artifact_root='artifacts/loss_landscape_analysis/flow_preconditioning/e4_debug',
    device=DEVICE,
    dtype='float32',
    seed=0,
    rho=1e-2,
    flow_steps=150,
    eval_every=15,
    flow_batch_size=8,
    flow_lr=1e-3,
    flow_grad_clip_norm=10.0,
    flow_num_layers=8,
    flow_hidden_dim=64,
    flow_network_depth=2,
    flow_log_scale_clamp=1.5,
    flow_random_samples=64,
    flow_trajectory_count=4,
    flow_trajectory_steps=8,
    heldout_geometry_samples=32,
    train_eval_samples=16,
    heldout_eval_samples=16,
)

DEBUG_CFG


## Build E4 Probe And Pools


In [ ]:
state = build_e4_debug_state(DEBUG_CFG)

print('output_dir:', state.output_dir)
print('device:', state.flow_pool.device)
print('flow_pool:', tuple(state.flow_pool.shape))
print('heldout_pool:', tuple(state.heldout_pool.shape))
print('train_eval_pool:', tuple(state.train_eval_pool.shape))
print('heldout_eval_pool:', tuple(state.heldout_eval_pool.shape))
print('probe output dim:', int(state.probe(state.flow_pool[0]).numel()))


## Train One NF And Evaluate Geometry During Training


In [ ]:
result = train_e4_debug_flow(state)
history = result.history
final_geometry = result.final_geometry

print('output_dir:', result.output_dir)
print('history rows:', len(history))
print('final geometry rows:', len(final_geometry))
display(history.tail(8))
display(final_geometry)


## Fast Sanity Checks


In [ ]:
step0 = history.iloc[0]
last = history.iloc[-1]

identity_train_abs = abs(float(step0['train_flow_R']) - float(step0['train_original_R']))
identity_heldout_abs = abs(float(step0['heldout_flow_R']) - float(step0['heldout_original_R']))
train_delta = float(last['train_R_delta'])
heldout_delta = float(last['heldout_R_delta'])
train_ratio = float(last['train_R_ratio'])
heldout_ratio = float(last['heldout_R_ratio'])

print(f"identity mismatch train_R={identity_train_abs:.6g} heldout_R={identity_heldout_abs:.6g}")
print(f"final train delta={train_delta:.6g} ratio={train_ratio:.4f}")
print(f"final heldout delta={heldout_delta:.6g} ratio={heldout_ratio:.4f}")

if identity_train_abs > 1e-3 or identity_heldout_abs > 1e-3:
    diagnosis = 'BUG: identity-initialized flow does not match original geometry. Check flow.inverse / probe_jacobians_for_flow wiring.'
elif heldout_delta < 0.0:
    diagnosis = 'OK: heldout geometry improved in this fast debug run.'
elif train_delta < 0.0 and heldout_delta >= 0.0:
    diagnosis = 'NF improves train-pool geometry but not heldout. This points to overfit / pool mismatch / objective generalization.'
else:
    diagnosis = 'NF does not even improve train-pool geometry. Debug optimizer, objective sign, flow capacity, or gradient path.'

Markdown(f'### Diagnosis
{diagnosis}')


## Geometry Curves


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

ax = axes[0, 0]
ax.plot(history['step'], history['train_flow_R'], label='train flow R')
ax.plot(history['step'], history['heldout_flow_R'], label='heldout flow R')
ax.axhline(float(history['train_original_R'].iloc[0]), linestyle='--', color='C0', alpha=0.5, label='train original R')
ax.axhline(float(history['heldout_original_R'].iloc[0]), linestyle='--', color='C1', alpha=0.5, label='heldout original R')
ax.set_xlabel('NF step')
ax.set_ylabel('global R, lower is better')
ax.set_yscale('log')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

ax = axes[0, 1]
plot_rows = history[history['step'] > 0]
ax.plot(plot_rows['step'], plot_rows['batch_loss'], label='train batch objective')
ax.set_xlabel('NF step')
ax.set_ylabel('batch objective')
ax.set_yscale('log')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(history['step'], history['train_flow_cond_median'], label='train flow cond median')
ax.plot(history['step'], history['heldout_flow_cond_median'], label='heldout flow cond median')
ax.set_xlabel('NF step')
ax.set_ylabel('pullback metric eig cond median')
ax.set_yscale('log')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

ax = axes[1, 1]
ax.plot(history['step'], history['train_flow_flow_displacement_median'], label='train displacement')
ax.plot(history['step'], history['heldout_flow_flow_displacement_median'], label='heldout displacement')
ax.set_xlabel('NF step')
ax.set_ylabel('median |flow(theta)-theta|')
ax.grid(True, alpha=0.25)
ax.legend(fontsize=8)

figure_path = Path(result.output_dir) / 'geometry_debug_curves.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('saved:', figure_path)


## Final Compact Table


In [ ]:
cols = [
    'step',
    'batch_loss',
    'train_original_R',
    'train_flow_R',
    'train_R_delta',
    'train_R_ratio',
    'heldout_original_R',
    'heldout_flow_R',
    'heldout_R_delta',
    'heldout_R_ratio',
    'train_flow_trace_cv',
    'heldout_flow_trace_cv',
    'train_flow_cond_median',
    'heldout_flow_cond_median',
    'train_flow_flow_cond_median',
    'heldout_flow_flow_cond_median',
]
compact = history[cols].copy()
display(compact)
compact.to_csv(Path(result.output_dir) / 'compact_history.csv', index=False)
print('saved:', Path(result.output_dir) / 'compact_history.csv')


## Files Written


In [ ]:
for path in sorted(Path(result.output_dir).iterdir()):
    print(path.name, path.stat().st_size)
